# 04c — Exploratory first-day-SOFA and ICU sensitivity analyses

`sofa_score` comes from `mimiciv_3_1_derived.first_day_sofa`. Its published definition can use information through 24 hours after ICU entry, while follow-up starts at ICU entry. All SOFA-containing core, ICU-adjusted and CHF/SOFA-stratified models are therefore exploratory sensitivity analyses that can use post-start information, not baseline-only severity adjustment. Temporal leakage or adjustment for post-exposure information may affect interpretation; its impact has not been quantified here. The primary age/sex model with era strata is unchanged. The separately executed [05c 24-hour landmark supplement](05c_time_aligned_severity.ipynb) is now reported from saved clinical aggregates. It uses a different population and follow-up origin; it does not reconstruct pre-ICU severity or independently verify individual SOFA component times. [05a](05a_patient_cluster_sensitivity.ipynb) separately compares ordinary and patient-cluster intervals for the unchanged core estimate.

The notebook reads the extended table created by `sql/04_build_outcomes_extended.sql`; it does not create that table. Run both outcome SQL scripts before 04a. Rare ICU types (<50 admissions, determined without outcomes) are pooled. Existing formulas, penalties and PH diagnostics are preserved, including additional exploratory CHF/SOFA strata.

[Timing limitations and separate reanalysis options](../docs/ANALYSIS_TIMING_LIMITATIONS.md)


<!-- Generated main figure guide: intro; edit scripts/render_main_figure_guides.py -->

## Saved clinical results

The tables and figures in this section display saved clinical execution results from [the public aggregate record](../docs/CURRENT_ANALYSIS.json). They do not represent a new clinical rerun. The separately labeled execution section describes how to reproduce outputs. Any synthetic execution is explicitly labeled and is not a clinical MIMIC result.

<!-- Generated main figure guide: core; edit scripts/figure_interpretation.py -->

### Figure 1 — Exploratory SOFA-core Cox estimates

**Data source:** Saved clinical execution results; [aggregate record](../docs/CURRENT_ANALYSIS.json).

![Figure 1 — Exploratory SOFA-core Cox estimates](../assets/cox_forest_plot.png)

**How to read**

- Each dot is an adjusted hazard ratio; its horizontal bar is an ordinary admission-level 95% confidence interval. Colors distinguish covariate rows, not statistical significance. The dashed line at HR=1 represents equal hazards under the fitted model. The horizontal scale is linear.
- Read the unit of each row: age is per year, SOFA is per point, male sex is relative to female sex, and the binary RAAS/CHF/CKD/diabetes rows compare flag 1 with flag 0. Different units mean that distance from 1 cannot be used to rank clinical importance across rows.

**What the results show**

- The RAAS row is HR 0.833 (95% CI 0.673–1.030) in the era-stratified, unpenalized SOFA-core model. Its interval includes 1, so it does not establish a clear association in this specification; it does not prove equal hazards or no association.

**Interpretation limits**

- This is the exploratory 04c model, not the refined primary 04a model or the separate patient-cluster analysis. First-day SOFA may include post-start information. The estimates are conditional associations and cannot establish treatment effects.

<!-- Generated main figure guide: sensitivity; edit scripts/figure_interpretation.py -->

### Figure 2 — Model sensitivity comparison

**Data source:** Saved clinical execution results; [aggregate record](../docs/CURRENT_ANALYSIS.json).

![Figure 2 — Model sensitivity comparison](../assets/cox_model_sensitivity_comparison.svg)

**How to read**

- Every row estimates the same qualifying-RAAS-orders versus no-qualifying-orders contrast under a different Cox specification. Dots are HRs and horizontal bars are that specification's original 95% intervals on a linear scale. Colors separate model rows for readability; these are not different medication groups.

**What the results show**

- The primary estimate is 0.732 (95% CI 0.592–0.904); the exploratory SOFA-core estimate is 0.833 (95% CI 0.673–1.030). The results depend on the specification, and several intervals include HR=1. These plotted intervals use the original inference, not the later patient-cluster intervals or 24-hour landmark population.

**Interpretation limits**

- L2 rows use penalization; the ICU and PH-stratified rows change covariates or baseline-hazard strata. Their differences cannot isolate a single confounder, and one interval crossing 1 while another does not is not a test of the difference between estimates.
- SOFA-containing rows are exploratory and can use post-start information. The set of models is a sensitivity analysis of an observational association, not independent replications of a causal treatment effect.

<!-- Generated main figure guide: runtime; edit scripts/render_main_figure_guides.py -->

## Runtime execution

The following cells run the shared refresh in an authorized BigQuery environment. They recompute the current models and exported figures; they are separate from the saved figures above.

In [ ]:
from pathlib import Path
import sys
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from current_analysis import refresh


In [ ]:
# Recompute current tables once; all figures, aggregate JSON and report source stay aligned.
results = refresh(REPO_ROOT)
print({"n": results["n"], "deaths": results["deaths"], "version": results["analysis_version"]})


In [ ]:
for key in ["core", "core_penalized", "icu_pooled", "icu_pooled_penalized"]:
    print(key, results["models"][key]["estimates"]["raas_pre_icu"])
results["rare_icu_rule"]


In [ ]:
# Exploratory models allow separate baseline hazards for CHF and exact SOFA.
# The ICU variant also stratifies on pooled ICU category. No CHF/SOFA coefficient is estimated.
for key in ["core_ph_stratified", "icu_ph_stratified"]:
    print(key, results["models"][key]["estimates"]["raas_pre_icu"])
    print("PH rank p-values:", results["models"][key]["ph_rank_p"])


In [ ]:
print("All four figures and current report source refreshed; render Quarto to update HTML.")
